# 🥇 CME Gold Futures (GC) Quantitative Research & AI Engine
## 🚀 Google Colab Master Execution Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/notebooks/intro.ipynb)

**Lead Architect:** Ayush Pandey &nbsp;|&nbsp; **Collaborator:** Bhaskar Pandey (`bhaskarauthor@gmail.com`)  
**Asset:** COMEX Gold Futures (`GC=F`, `MGC=F`) & CME Globex MDP 3.0 Microstructure Lake  

---

### 📌 What This Notebook Does
1. **Zero-Copy Google Drive Mount:** Connects directly to this repository inside your Google Drive (`MyDrive/Databento_Gold_Futures_Research`).
2. **Hardware Acceleration:** Uses Google Colab's free **NVIDIA Tesla T4 GPU (16 GB VRAM)** with PyTorch CUDA and FP16 Tensor Cores.
3. **Microstructure Alpha Mining:** Computes Level-1 Order Book Imbalance (OBI), Volume-Weighted Micro-Price, and Multi-Horizon Information Coefficients ($k \in \{1, 3, 5, 10, 20\}$ ticks).
4. **Deep Neural Network Execution:** Runs high-frequency BiLSTM-Attention and SOTA DeepLOB inference/backtests.
5. **Algorithmic Strategy Backtest:** Simulates Passive Maker, Aggressive Taker, and Benchmark Midpoint execution regimes.

## 🔌 Step 1: Mount Google Drive & Set Workspace
Since this entire codebase is already stored in your Google Drive, mounting Drive allows Colab to access all data, models, and scripts directly without downloading anything to your local machine.

In [ ]:
import os
import sys
from pathlib import Path

# Check if running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("[*] Mounting Google Drive...")
    drive.mount('/content/drive')
    
    # Locate repository on Google Drive
    repo_candidates = [
        Path('/content/drive/MyDrive/Databento_Gold_Futures_Research'),
        Path('/content/drive/My Drive/Databento_Gold_Futures_Research')
    ]
    repo_root = None
    for p in repo_candidates:
        if p.exists():
            repo_root = p
            break
            
    if repo_root:
        os.chdir(str(repo_root))
        print(f"[✓] Successfully changed working directory to: {os.getcwd()}")
    else:
        print("[!] Repository folder not found under default Google Drive path.")
        print("Current contents of /content/drive/MyDrive:")
        print(os.listdir('/content/drive/MyDrive')[:10])
else:
    print(f"[✓] Running locally in: {os.getcwd()}")

## ⚙️ Step 2: Install Dependencies & Check GPU Acceleration

In [ ]:
# Install quantitative analysis packages quietly
!pip install -q pyarrow databento duckdb zstandard matplotlib

import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"[✓] CUDA GPU Active:   {torch.cuda.get_device_name(0)}")
    print(f"[✓] Total GPU VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[i] Compute: CPU Mode (Go to Runtime -> Change runtime type -> Select T4 GPU for 10x speedup)")
print("=" * 60)

## 📊 Step 3: Load CME Gold Order Book Data (MBP-1 Schema)
We load Level 1 Market-By-Price (`mbp-1`) data and inspect Top of Book (BBO: Best Bid & Ask).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = Path("data/GC_sample_mbp1.parquet")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path.resolve()}")

df = pd.read_parquet(data_path)
print(f"[✓] Total Order Book Events: {len(df):,}")
print(f"[✓] Columns available: {list(df.columns)}")
df[["ts_recv", "action", "side", "price", "size", "bid_px_00", "ask_px_00", "bid_sz_00", "ask_sz_00"]].head(5)

## 🔬 Step 4: Engineer Microstructure Alpha Signals
- **Normalized Order Book Imbalance (OBI):**
  $$\text{OBI}_t = \frac{Q_t^{\text{bid}} - Q_t^{\text{ask}}}{Q_t^{\text{bid}} + Q_t^{\text{ask}}} \in [-1.0, +1.0]$$
- **Volume-Weighted Micro-Price:**
  $$P_t^{\text{micro}} = \frac{Q_t^{\text{bid}} P_t^{\text{ask}} + Q_t^{\text{ask}} P_t^{\text{bid}}}{Q_t^{\text{bid}} + Q_t^{\text{ask}}}$$
- **Spread in Basis Points (bps):**
  $$\text{Spread (bps)} = \frac{P_t^{\text{ask}} - P_t^{\text{bid}}}{P_t^{\text{mid}}} \times 10,000$$

In [ ]:
# Compute microstructure features
mid = (df["bid_px_00"] + df["ask_px_00"]) / 2.0
tot_sz = df["bid_sz_00"] + df["ask_sz_00"]

df["mid_price"] = mid
df["obi"] = np.where(tot_sz > 0, (df["bid_sz_00"] - df["ask_sz_00"]) / tot_sz, 0.0)
df["micro_price"] = np.where(tot_sz > 0, (df["bid_sz_00"] * df["ask_px_00"] + df["ask_sz_00"] * df["bid_px_00"]) / tot_sz, mid)
df["micro_spread"] = df["micro_price"] - mid
df["spread_bps"] = ((df["ask_px_00"] - df["bid_px_00"]) / mid) * 10000.0

print(f"Average Mid-Price:    ${df['mid_price'].mean():,.2f}")
print(f"Average Spread (bps): {df['spread_bps'].mean():.2f} bps")
print(f"Average OBI:          {df['obi'].mean():+.4f}")

# Visualize OBI & Micro-Price Dynamics
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
sample_slice = df.iloc[:400]

axes[0].plot(sample_slice.index, sample_slice["mid_price"], label="Mid Price", color="gold", linewidth=1.5)
axes[0].plot(sample_slice.index, sample_slice["micro_price"], label="Micro Price", color="cyan", linestyle="--", linewidth=1.2)
axes[0].set_ylabel("Price ($/oz)")
axes[0].set_title("CME Gold Futures: Mid-Price vs Volume-Weighted Micro-Price (First 400 Ticks)")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

axes[1].bar(sample_slice.index, sample_slice["obi"], color=np.where(sample_slice["obi"] >= 0, "green", "red"), alpha=0.6, width=1.0)
axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_ylabel("OBI [-1, +1]")
axes[1].set_xlabel("Tick Event Index")
axes[1].set_title("Normalized Order Book Imbalance (OBI)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📈 Step 5: Multi-Horizon Information Coefficient (IC) Decay
Evaluate whether Order Book Imbalance (OBI) and Micro-Price spread reliably predict forward price movements across $k \in \{1, 3, 5, 10, 20\}$ tick horizons.

In [ ]:
horizons = [1, 3, 5, 10, 20]
ic_results = []

for k in horizons:
    df[f"fwd_ret_{k}"] = (df["mid_price"].shift(-k) - df["mid_price"]) / df["mid_price"]
    sub = df[["obi", "micro_spread", f"fwd_ret_{k}"]].dropna()
    
    p_obi = sub["obi"].corr(sub[f"fwd_ret_{k}"])
    p_micro = sub["micro_spread"].corr(sub[f"fwd_ret_{k}"])
    s_micro = sub["micro_spread"].rank().corr(sub[f"fwd_ret_{k}"].rank())
    
    ic_results.append({
        "Horizon (k Ticks)": f"k = {k}",
        "OBI Pearson IC": round(p_obi, 4),
        "Micro-Price Pearson IC": round(p_micro, 4),
        "Micro-Price Spearman Rank IC": round(s_micro, 4),
    })

df_ic = pd.DataFrame(ic_results)
print("📊 Predictive Information Coefficient (IC) Matrix:")
df_ic

## 🧠 Step 6: Deep Learning GPU Model Inference & Evaluation
We load the trained PyTorch BiLSTM-Attention model checkpoint (`models/gold_orderbook_t4_model.pt`) and run GPU inference to evaluate directional accuracy.

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# BiLSTM-Attention Model Definition
class DeepOrderBookNet(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=64, num_layers=2):
        super(DeepOrderBookNet, self).__init__()
        self.bilstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True
        )
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim * 2, num_heads=4, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.LayerNorm(64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        pooled = attn_out[:, -1, :]
        return self.fc(pooled)

# Checkpoint path
ckpt = Path("models/gold_orderbook_t4_model.pt")
if ckpt.exists():
    model = DeepOrderBookNet().to(device)
    model.load_state_dict(torch.load(str(ckpt), map_location=device))
    model.eval()
    print(f"[✓] Successfully loaded model from: {ckpt}")
    print(f"[✓] Model architecture initialized on: {device}")
else:
    print(f"[!] Checkpoint not found at {ckpt}. You can run scripts/train_colab_t4_gpu.py to train a fresh model.")

## ⚡ Step 7: Run Microstructure Backtest or Platform Scripts
You can execute any platform script or CLI tool directly from Colab:

In [ ]:
# Execute the GPU Model Backtest
!python scripts/backtest_colab_t4_gpu.py

In [ ]:
# Inspect Quantitative Status & Health
!python manage.py pnl